# Ru’ya — VisDrone YOLOv8 Training

This notebook prepares VisDrone annotations for person-only detection, trains a YOLOv8 model, evaluates it, and optionally creates a demo video. Download the dataset separately and set `DATASET_PATH` before running. Model weights and generated media are created locally and are not committed to this repository.


In [ ]:
!pip install ultralytics opencv-python pillow -q

In [ ]:
from pathlib import Path
import os
import shutil
import cv2
from PIL import Image
from ultralytics import YOLO

# Set DATASET_PATH to the folder containing the VisDrone2019-DET-* directories.
# Example (Kaggle): Path("/kaggle/input/visdrone-dataset")
DATASET_PATH = Path(os.environ.get("VISDRONE_DATASET_PATH", "data/visdrone-dataset"))
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"VisDrone dataset not found at {DATASET_PATH}. "
        "Download the dataset from its official source and update DATASET_PATH."
    )

OUTPUT_ROOT = Path("visdrone")
(OUTPUT_ROOT / "images/train").mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "images/val").mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "labels/train").mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "labels/val").mkdir(parents=True, exist_ok=True)

In [ ]:
def convert_visdrone_to_yolo(label_path, save_path, img_width, img_height):

    with open(label_path, 'r') as f:
        lines = f.readlines()

    yolo_lines = []

    for line in lines:
        parts = line.strip().split(',')

        if len(parts) < 6:
            continue

        x, y, w, h, score, class_id = map(int, parts[:6])

        if class_id != 1:   # keep PERSON only
            continue

        x_center = (x + w/2) / img_width
        y_center = (y + h/2) / img_height
        w_norm = w / img_width
        h_norm = h / img_height   # ✅ THIS IS THE FIX

        yolo_lines.append(f"0 {x_center} {y_center} {w_norm} {h_norm}\n")

    with open(save_path, 'w') as f:
        f.writelines(yolo_lines)

In [ ]:
def process_split(image_dir, label_dir, out_img_dir, out_lbl_dir):

    for img_name in os.listdir(image_dir):

        img_path = os.path.join(image_dir, img_name)
        label_name = img_name.replace(".jpg",".txt")
        label_path = os.path.join(label_dir, label_name)

        if not os.path.exists(label_path):
            continue

        img = Image.open(img_path)
        w, h = img.size

        shutil.copy(img_path, os.path.join(out_img_dir, img_name))

        convert_visdrone_to_yolo(
            label_path,
            os.path.join(out_lbl_dir, label_name),
            w,
            h
        )

In [ ]:
process_split(
    str(DATASET_PATH / "VisDrone2019-DET-train/VisDrone2019-DET-train/images"),
    str(DATASET_PATH / "VisDrone2019-DET-train/VisDrone2019-DET-train/annotations"),
    "visdrone/images/train",
    "visdrone/labels/train"
)

In [ ]:
process_split(
    str(DATASET_PATH / "VisDrone2019-DET-val/VisDrone2019-DET-val/images"),
    str(DATASET_PATH / "VisDrone2019-DET-val/VisDrone2019-DET-val/annotations"),
    "visdrone/images/val",
    "visdrone/labels/val"
)

In [ ]:
with open("visdrone.yaml", "w") as f:
    f.write("""
path: visdrone
train: images/train
val: images/val

names:
  0: person
""")

In [ ]:
model = YOLO("yolov8s.pt")

model.train(
    data="visdrone.yaml",
    epochs=30,
    imgsz=1280,
    batch=8,
    optimizer="AdamW",
    lr0=0.0008,
    mosaic=1.0,
    mixup=0.1,
    degrees=5,
    scale=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    device=0,
    name="final_drone_model"
)

In [ ]:
BEST_MODEL_PATH = Path("runs/detect/final_drone_model/weights/best.pt")
if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Trained weights not found at {BEST_MODEL_PATH}. "
        "Run the training cell first and confirm the run name."
    )
model = YOLO(str(BEST_MODEL_PATH))

In [ ]:
metrics = model.val()

print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP@50:", metrics.box.map50)
print("mAP@50-95:", metrics.box.map)

In [ ]:
model.export(format="onnx")

In [ ]:
# Optional: create a demo video from the VisDrone test-dev images.
TEST_IMAGES = DATASET_PATH / "VisDrone2019-DET-test-dev/VisDrone2019-DET-test-dev/images"
if not TEST_IMAGES.exists():
    raise FileNotFoundError(f"Test images folder not found: {TEST_IMAGES}")

images = sorted(p for p in TEST_IMAGES.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
if not images:
    raise FileNotFoundError(f"No images found in {TEST_IMAGES}")

VIDEO_WIDTH, VIDEO_HEIGHT = 1280, 720
video_path = "drone_test.mp4"
out = cv2.VideoWriter(video_path, cv2.VideoWriter_fourcc(*"mp4v"), 10, (VIDEO_WIDTH, VIDEO_HEIGHT))
for image_path in images[:200]:
    img = cv2.imread(str(image_path))
    if img is None:
        continue
    out.write(cv2.resize(img, (VIDEO_WIDTH, VIDEO_HEIGHT)))
out.release()
print(f"Demo video saved to {video_path}")

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import time
from IPython.display import Video

model = YOLO(str(BEST_MODEL_PATH))

cap = cv2.VideoCapture("drone_test.mp4")

GRID_ROWS = 6
GRID_COLS = 6

T1 = 2
T2 = 4

VIDEO_WIDTH = 1280
VIDEO_HEIGHT = 720

out = cv2.VideoWriter(
    "crowd_output.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    10,
    (VIDEO_WIDTH, VIDEO_HEIGHT)
)

frame_count = 0
start_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (VIDEO_WIDTH, VIDEO_HEIGHT))

    # Detect persons
    results = model.predict(frame, imgsz=640, conf=0.3, verbose=False)
    boxes = results[0].boxes.xyxy.cpu().numpy()

    h, w = frame.shape[:2]

    cell_w = w // GRID_COLS
    cell_h = h // GRID_ROWS
    cell_area = cell_w * cell_h     # A in the formula

    density_map = np.zeros((GRID_ROWS, GRID_COLS))

    # Count detections per cell (N)
    for box in boxes:
        x1, y1, x2, y2 = map(int, box)

        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)

        gx = cx // cell_w
        gy = cy // cell_h

        if gx < GRID_COLS and gy < GRID_ROWS:
            density_map[gy, gx] += 1

    # Compute Density + Risk
    for i in range(GRID_ROWS):
        for j in range(GRID_COLS):

            N = density_map[i, j]      # detections
            D = N         # D = N / A  ✅

            # Risk Classification (from slide)
            if D < T1:
                color = (0,255,0)      # Low Risk (Green)
                label = "LOW"
            elif T1 <= D < T2:
                color = (0,255,255)    # Medium Risk (Yellow)
                label = "MED"
            else:
                color = (0,0,255)      # High Risk (Red)
                label = "HIGH"

            x1 = j * cell_w
            y1 = i * cell_h
            x2 = x1 + cell_w
            y2 = y1 + cell_h

            cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
            cv2.putText(frame,label,(x1+5,y1+20),
                        cv2.FONT_HERSHEY_SIMPLEX,0.6,color,2)

    total_people = len(boxes)

    frame_count += 1
    fps = frame_count / (time.time() - start_time)

    cv2.putText(frame,f"People: {total_people}",(20,40),
                cv2.FONT_HERSHEY_SIMPLEX,1,(255,0,0),2)

    cv2.putText(frame,f"FPS: {fps:.2f}",(20,80),
                cv2.FONT_HERSHEY_SIMPLEX,1,(255,255,0),2)

    out.write(frame)

cap.release()
out.release()

print("Processing Done!")
print("Average FPS:", fps)

In [ ]:
Video("crowd_output.mp4")